In [ ]:
import pandas as pd
from bs4 import BeautifulSoup

import time
import random
import os

from selenium import webdriver

from urllib.parse import urlparse
import hashlib #TODO precisa padronizar isso em algum step



In [18]:
os.makedirs("raw_html", exist_ok=True)

# Codigos gerais

In [28]:
def salvar_pagina_atual(driver, nome_arquivo=None, subpasta=None):
    if nome_arquivo is None:
        path = urlparse(driver.current_url).path.strip("/")
        nome_arquivo = (path.replace("/", "_") or "index") + ".html"

    if subpasta:
        caminho = os.path.join(f"raw_html/{subpasta}", nome_arquivo)
    else:
        caminho = os.path.join("raw_html", nome_arquivo)
    with open(caminho, "w", encoding="utf-8") as f:
        f.write(driver.page_source)  # page_source = HTML já renderizado (pós-JS)

    print(f"Página salva em: {caminho}")
    return caminho

# Dados de aba `brazil`

In [ ]:
from bs4 import BeautifulSoup

with open(caminho_salvo, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

registros = []
for a in soup.select("table tbody tr td:first-child a[href^='/brazil/']"):
    registros.append({"nome": a.get_text(strip=True), "href": a.get("href")})

print(len(registros))

37


[{'nome': 'Porto Alegre', 'href': '/brazil/porto-alegre/'},
 {'nome': 'Sao Paulo', 'href': '/brazil/sao-paulo/'},
 {'nome': 'Tambore', 'href': '/brazil/tambore/'},
 {'nome': 'Campinas', 'href': '/brazil/campinas/'},
 {'nome': 'Rio de Janeiro', 'href': '/brazil/rio-de-janeiro/'}]

# Pegando informaçao 20 datacenter selecionados

In [ ]:
# def gerar_id(link: str) -> str:
#     return 'dc_' + hashlib.md5(link.encode('utf-8')).hexdigest()[:10]

# df.insert(0, 'id_datacenter', df['link_datacenter'].apply(gerar_id))

In [40]:
df = pd.read_csv('20_datacenters_selectes_link_datacentermap.csv')

In [ ]:
PASTA_DESTINO = "datacentermap_20"
os.makedirs(f"raw_html/{PASTA_DESTINO}", exist_ok=True)

In [ ]:
DELAY_MIN = 30
DELAY_MAX = 60

registros = df.to_dict("records")

In [44]:
driver = webdriver.Chrome()

for i, registro in enumerate(registros, 1):
    id_datacenter = registro["id_datacenter"]
    link = registro["link_datacenter"]

    # --- overview ---
    print(f"[{i}/{len(registros)}] Abrindo overview -> {link}")
    driver.get(link)
    nome_arquivo_overview = f"{id_datacenter}_overview.html"
    salvar_pagina_atual(driver, nome_arquivo=nome_arquivo_overview, subpasta=PASTA_DESTINO)

    espera = random.uniform(DELAY_MIN, DELAY_MAX)
    print(f"  Aguardando {espera:.1f}s antes da próxima...")
    time.sleep(espera)

    # --- specs ---
    link_specs = link.rstrip("/") + "/specs/"
    print(f"[{i}/{len(registros)}] Abrindo specs -> {link_specs}")
    driver.get(link_specs)
    nome_arquivo_specs = f"{id_datacenter}_specs.html"
    salvar_pagina_atual(driver, nome_arquivo=nome_arquivo_specs, subpasta=PASTA_DESTINO)

    if i < len(registros):
        espera = random.uniform(DELAY_MIN, DELAY_MAX)
        print(f"  Aguardando {espera:.1f}s antes da próxima...")
        time.sleep(espera)

print(f"Concluído! Páginas salvas em raw_html/{PASTA_DESTINO}/")

[1/1] Abrindo overview -> https://www.datacentermap.com/brazil/fortaleza/casa-dos-ventos-data-center/
Página salva em: raw_html/datacentermap_20\dc_bc34fbde0a_overview.html
  Aguardando 30.3s antes da próxima...
[1/1] Abrindo specs -> https://www.datacentermap.com/brazil/fortaleza/casa-dos-ventos-data-center/specs/
Página salva em: raw_html/datacentermap_20\dc_bc34fbde0a_specs.html
Concluído! Páginas salvas em raw_html/datacentermap_20/


# lendo os arquivos html nivel_estado

In [37]:
import json
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

RAW_HTML_DIR = Path("raw_html")  # ajuste se o notebook não estiver em notebooks/extract_raw_data


def extract_datacenters(path: Path) -> list[dict]:
    html = path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")

    tag = soup.find("script", id="__NEXT_DATA__")
    if tag is None or tag.string is None:
        return []

    payload = json.loads(tag.string)
    dcs = payload["props"]["pageProps"]["mapdata"]["dcs"]

    rows = []
    for feature in dcs:
        p = feature["properties"]
        rows.append({
            "source_file": path.name,
            "name": p.get("name"),
            "company": p.get("companyname"),
            "address": p.get("address"),
            "postal_code": p.get("postal"),
            "city": p.get("city"),
            "state": p.get("state"),
            "country": p.get("country"),
            "url": f"https://www.datacentermap.com{p.get('url')}" if p.get("url") else None,
        })
    return rows

In [41]:
all_rows = []
for path in sorted(RAW_HTML_DIR.glob("brazil_*.html")):
    file_rows = extract_datacenters(path)
    print(f"{path.name}: {len(file_rows)} data centers")
    all_rows.extend(file_rows)

df = pd.DataFrame(all_rows).drop_duplicates(subset=["url"]).reset_index(drop=True)
print(f"\nTotal: {len(df)} data centers únicos")

brazil_aracaju.html: 1 data centers
brazil_ararangu.html: 1 data centers
brazil_belem.html: 1 data centers
brazil_belo-horizonte.html: 4 data centers
brazil_blumenau.html: 2 data centers
brazil_brasilia.html: 11 data centers
brazil_campina-grande.html: 2 data centers
brazil_campinas.html: 37 data centers
brazil_campo-grande.html: 2 data centers
brazil_cascavel.html: 2 data centers
brazil_cravinhos.html: 1 data centers
brazil_curitiba.html: 7 data centers
brazil_fortaleza.html: 13 data centers
brazil_goiania.html: 2 data centers
brazil_itauna.html: 1 data centers
brazil_joao-pessoa.html: 1 data centers
brazil_joinville.html: 2 data centers
brazil_lagoa-da-prata.html: 1 data centers
brazil_manaus.html: 2 data centers
brazil_maringa.html: 3 data centers
brazil_muriae.html: 1 data centers
brazil_palmas.html: 2 data centers
brazil_parnaiba.html: 2 data centers
brazil_porto-alegre.html: 15 data centers
brazil_recife.html: 3 data centers
brazil_ribeirao-preto.html: 1 data centers
brazil_rio-d

In [39]:
df.to_csv("datacenters_brazil.csv", index=False, encoding="utf-8-sig")

In [ ]:
[df['url'] + '/specs/']

0      https://www.datacentermap.com/brazil/aracaju/p...
1      https://www.datacentermap.com/brazil/ararangu/...
2      https://www.datacentermap.com/brazil/belem/ele...
3      https://www.datacentermap.com/brazil/belo-hori...
4      https://www.datacentermap.com/brazil/belo-hori...
                             ...                        
236    https://www.datacentermap.com/brazil/vitoria/n...
237    https://www.datacentermap.com/brazil/vitoria/n...
238    https://www.datacentermap.com/brazil/vitoria/v...
239    https://www.datacentermap.com/brazil/xaxim/vis...
240    https://www.datacentermap.com/brazil/xaxim/fer...
Name: url, Length: 241, dtype: str

# atribuindo specs

In [3]:
df = pd.read_csv('datacenters_brazil.csv')

In [4]:
df.dropna(subset=['url'], inplace=True)

In [5]:
df['url_specs'] = df['url'].apply(lambda x: x + 'specs/')

# baixando os specs de cada datacenter

In [ ]:

DELAY_MIN = 30
DELAY_MAX = 60

lista = list(df['url_specs'])

tamanho_bloco = 11

for i in range(0, len(lista), tamanho_bloco):
    bloco = lista[i : i + tamanho_bloco]

    driver = webdriver.Chrome()
    for url in bloco:
    
        print(f" Abrindo {url}")
        driver.get(url)
        nome_arquivo = 'brazil_' + url.split('/')[-3] +'.html'

        caminho = salvar_pagina_atual(driver, nome_arquivo=nome_arquivo, subpasta='specs')
        espera = random.uniform(DELAY_MIN, DELAY_MAX)
        print(f"  Aguardando {espera:.1f}s antes da próxima...")
        time.sleep(espera)
    driver.quit()

print("Concluído! Todas as páginas foram salvas em raw_html/specs")


 Abrindo https://www.datacentermap.com/brazil/aracaju/pix-g8-aracaju/specs/
Página salva em: raw_html/specs\brazil_pix-g8-aracaju.html
  Aguardando 40.7s antes da próxima...
 Abrindo https://www.datacentermap.com/brazil/ararangu/contato-cloud/specs/
Página salva em: raw_html/specs\brazil_contato-cloud.html
  Aguardando 56.3s antes da próxima...
 Abrindo https://www.datacentermap.com/brazil/belem/elea-bel1/specs/
Página salva em: raw_html/specs\brazil_elea-bel1.html
  Aguardando 57.5s antes da próxima...
 Abrindo https://www.datacentermap.com/brazil/belo-horizonte/ativas-datacenter-sa/specs/
Página salva em: raw_html/specs\brazil_ativas-datacenter-sa.html
  Aguardando 39.4s antes da próxima...
 Abrindo https://www.datacentermap.com/brazil/belo-horizonte/century-cem-dc01/specs/
Página salva em: raw_html/specs\brazil_century-cem-dc01.html
  Aguardando 42.8s antes da próxima...
 Abrindo https://www.datacentermap.com/brazil/belo-horizonte/master-da-web/specs/
Página salva em: raw_html/specs

In [25]:
nome_arquivo

'brazil_contato-cloud.html'

In [8]:
df['url_specs'].to_list()[100]

'https://www.datacentermap.com/brazil/porto-alegre/quantico-canoas/specs/'

In [10]:
df[df['url_specs'] == 'https://www.datacentermap.com/brazil/ribeirao-preto/convex-idc-rpo/specs/']

,source_file,name,company,address,postal_code,city,state,country,url,url_specs
116,brazil_ribeirao-preto.html,Convex IDC Ribeirão Preto,Convex Internet Solutions,"R. Bernardino de Campos, 1001",14015-130,Ribeirao Preto,SP,Brazil,https://www.datacentermap.com/brazil/ribeirao-...,https://www.datacentermap.com/brazil/ribeirao-...


In [25]:
df[df['url_specs'].apply(lambda x: 'data-center-cachoeirinha' in x)]

,source_file,name,company,address,postal_code,city,state,country,url,url_specs
102,brazil_porto-alegre.html,Parks Data Center Cachoeirinha,Parks S/A Comunicações Digitais,"Av. Cruzeiro, 530 - Distrito Industrial",94930-615,Cachoeirinha,NaN,Brazil,https://www.datacentermap.com/brazil/porto-ale...,https://www.datacentermap.com/brazil/porto-ale...
